In [ ]:
from pathlib import Path
import os

# Set PROJECT_DATA_DIR before launching Jupyter to use data stored elsewhere.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".gitignore").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = Path(os.environ.get("PROJECT_DATA_DIR", str(PROJECT_ROOT / "data"))).expanduser()


# LightGBM Online-Learning Simulation for Fraud Detection

This notebook converts the existing batch LightGBM fraud workflow into a **time-ordered streaming demo**.

What it does:
- trains on an initial historical block of transactions
- holds out the rest as a simulated live stream
- scores each incoming chunk **before** seeing labels
- then reveals labels and retrains periodically
- logs how predictions and performance evolve over time

Important note:
- **LightGBM is not a true `partial_fit` model**
- this notebook simulates online learning with **chunked time-based retraining**
- that is usually the cleanest and most defensible demo for a “live fraud stream”


## 1) Imports and configuration

In [ ]:

import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import clear_output, display
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_fscore_support
import lightgbm as lgb

warnings.filterwarnings("ignore")

In [ ]:
INITIAL_TRAIN_FRACTION = 0.70   # first 70% of time-ordered data = historical training block
STREAM_FRACTION = 0.30          # remaining 30% = simulated live stream

CHUNK_SIZE = 1000               # number of "new transactions" per incoming batch
RETRAIN_EVERY_CHUNKS = 1        # retrain after every chunk for a more visibly live demo
MAX_STREAM_CHUNKS = 8           # reduce for a faster demo; set to None to run the full stream
TRAINING_WINDOW = 100_000       # keep most recent rows for retraining; set None for expanding window

FRAUD_THRESHOLD = 0.50          # probability threshold used for predicted class in the live display
SLEEP_SECONDS = 0.0             # add e.g. 0.5 for a dramatic live demo feel

RANDOM_STATE = 42

## 2) Load the dataset

This cell looks for one of these files in the current folder:

- `train_merged_submission.csv`
- `train_merged.csv`

If neither exists, it falls back to a small synthetic fraud-like dataset so the notebook still runs for rehearsal.


In [ ]:

from sklearn.datasets import make_classification

candidate_paths = [
    Path(str(DATA_DIR / 'train_merged_submission.csv')),
    Path(str(DATA_DIR / 'train_merged.csv')),
    Path(str(DATA_DIR / 'train_merged_submission.csv')),
    Path(str(DATA_DIR / 'train_merged.csv')),
]

data_path = None
for p in candidate_paths:
    if p.exists():
        data_path = p
        break

if data_path is not None:
    df = pd.read_csv(data_path)
    using_synthetic = False
    print(f"Loaded real dataset: {data_path} | shape={df.shape}")
else:
    X_syn, y_syn = make_classification(
        n_samples=25000,
        n_features=40,
        n_informative=12,
        n_redundant=8,
        n_clusters_per_class=2,
        weights=[0.985, 0.015],
        flip_y=0.003,
        random_state=RANDOM_STATE,
    )
    df = pd.DataFrame(X_syn, columns=[f"V{i}" for i in range(X_syn.shape[1])])
    df["TransactionDT"] = np.arange(len(df))
    df["TransactionAmt"] = np.abs(np.random.normal(100, 40, size=len(df)))
    df["ProductCD"] = np.random.choice(["W", "C", "H", "R", "S"], size=len(df))
    df["card4"] = np.random.choice(["visa", "mastercard", "discover", "american express"], size=len(df))
    df["isFraud"] = y_syn
    using_synthetic = True
    print("No local CSV found. Using synthetic fraud-like data for a runnable rehearsal demo.")
    print(f"Synthetic shape={df.shape}")


## 3) Preprocessing helpers

These helpers keep the preprocessing **train-only** so the stream is not leaking future information into the initial model.


In [ ]:

def basic_feature_cleanup(raw_df: pd.DataFrame) -> tuple[pd.DataFrame, list[str], list[str]]:
    df = raw_df.copy()

    if "isFraud" not in df.columns:
        raise ValueError("Dataset must contain an 'isFraud' column.")

    # Drop very high-missing columns
    missing_ratio = df.isnull().mean()
    df = df.loc[:, missing_ratio < 0.95].copy()

    # Drop constant / near-constant columns
    low_var_cols = [col for col in df.columns if df[col].nunique(dropna=False) <= 1]
    df = df.drop(columns=low_var_cols, errors="ignore")

    # Identify categorical vs numerical
    categorical = []
    numerical = []

    for col in df.columns:
        if col == "isFraud":
            continue

        nunique = df[col].nunique(dropna=True)

        if df[col].dtype == "object":
            categorical.append(col)
        elif nunique < 50:
            categorical.append(col)
        else:
            numerical.append(col)

    # Domain-style corrections from your original notebook
    known_cat = ["card", "addr", "email", "Product", "M"]
    for col in df.columns:
        if col != "isFraud" and any(k in col for k in known_cat):
            categorical.append(col)

    categorical = sorted(list(set([c for c in categorical if c in df.columns and c != "isFraud"])))
    numerical = sorted([c for c in df.columns if c not in categorical + ["isFraud"]])

    return df, categorical, numerical


def fit_preprocessor(train_df: pd.DataFrame, categorical_cols: list[str], numerical_cols: list[str]):
    train_df = train_df.copy()
    mappings = {}
    medians = {}

    # categorical: fill missing with "unknown", fit category mapping on train only
    for col in categorical_cols:
        train_df[col] = train_df[col].astype(str).fillna("unknown")
        categories = pd.Series(train_df[col].astype(str).unique())
        mappings[col] = {k: i for i, k in enumerate(categories)}

    # numerical: fit median on train only
    for col in numerical_cols:
        medians[col] = train_df[col].median()

    return {"categorical": categorical_cols, "numerical": numerical_cols, "mappings": mappings, "medians": medians}


def transform_with_preprocessor(df_in: pd.DataFrame, preprocessor: dict) -> pd.DataFrame:
    df = df_in.copy()
    categorical_cols = preprocessor["categorical"]
    numerical_cols = preprocessor["numerical"]
    mappings = preprocessor["mappings"]
    medians = preprocessor["medians"]

    # numerical features + missingness indicators
    for col in numerical_cols:
        df[f"{col}_missing"] = df[col].isna().astype("int8")
        df[col] = df[col].fillna(medians[col])

    # categorical features
    for col in categorical_cols:
        df[col] = df[col].astype(str).fillna("unknown")
        df[col] = df[col].map(mappings[col]).fillna(-1).astype("int32")

    return df


def build_lgbm(scale_pos_weight: float) -> lgb.LGBMClassifier:
    return lgb.LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=64,
        max_depth=-1,
        min_child_samples=50,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        objective="binary",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        scale_pos_weight=scale_pos_weight,
        verbosity=-1,
    )


def train_model(train_raw: pd.DataFrame, feature_cols: list[str], categorical_cols: list[str], numerical_cols: list[str]):
    train_raw = train_raw.copy()

    preprocessor = fit_preprocessor(
        train_df=train_raw[feature_cols].copy(),
        categorical_cols=[c for c in categorical_cols if c in feature_cols],
        numerical_cols=[c for c in numerical_cols if c in feature_cols],
    )

    X_train = transform_with_preprocessor(train_raw[feature_cols].copy(), preprocessor)
    y_train = train_raw["isFraud"].astype(int).copy()

    pos = max((y_train == 1).sum(), 1)
    neg = max((y_train == 0).sum(), 1)
    scale_pos_weight = neg / pos

    model = build_lgbm(scale_pos_weight=scale_pos_weight)
    model.fit(X_train, y_train)

    return model, preprocessor, X_train.columns.tolist()


## 4) Sort by time and create the historical block + live stream

In [ ]:

df, categorical_cols, numerical_cols = basic_feature_cleanup(df)

# Sort by transaction time if available; otherwise preserve current order
if "TransactionDT" in df.columns:
    df = df.sort_values("TransactionDT").reset_index(drop=True)
else:
    df = df.reset_index(drop=True)
    df["TransactionDT"] = np.arange(len(df))

# Avoid training on row identifiers
drop_candidates = ["isFraud", "TransactionID"]
feature_cols = [c for c in df.columns if c not in drop_candidates]

split_idx = int(len(df) * INITIAL_TRAIN_FRACTION)
historical_df = df.iloc[:split_idx].copy()
stream_df = df.iloc[split_idx:].copy()

print("Rows:", len(df))
print("Historical block:", historical_df.shape)
print("Live stream block:", stream_df.shape)
print("Fraud rate (historical):", round(historical_df["isFraud"].mean(), 4))
print("Fraud rate (stream):", round(stream_df["isFraud"].mean(), 4))


## 5) Initial training on historical data

In [ ]:

model, preprocessor, transformed_columns = train_model(
    train_raw=historical_df,
    feature_cols=feature_cols,
    categorical_cols=categorical_cols,
    numerical_cols=numerical_cols,
)

print("Initial model trained.")
print("Training rows used:", len(historical_df))
print("Number of model features after preprocessing:", len(transformed_columns))


## 6) Simulate live transactions arriving in time order

How the loop works:
1. a new chunk arrives
2. the current model scores it immediately
3. the fraud labels are then “revealed”
4. the chunk is appended to the training history
5. the model is retrained after the configured number of chunks

This is the core of the online-learning simulation.


In [ ]:

results = []
latest_examples = []

adaptive_train_df = historical_df.copy()
all_seen_stream_rows = 0
retrain_count = 0

num_chunks_total = int(np.ceil(len(stream_df) / CHUNK_SIZE))
if MAX_STREAM_CHUNKS is not None:
    num_chunks_total = min(num_chunks_total, MAX_STREAM_CHUNKS)

for chunk_idx in range(num_chunks_total):
    start = chunk_idx * CHUNK_SIZE
    end = min((chunk_idx + 1) * CHUNK_SIZE, len(stream_df))
    chunk_raw = stream_df.iloc[start:end].copy()

    if len(chunk_raw) == 0:
        break

    # Score incoming transactions BEFORE the labels are used
    X_chunk = transform_with_preprocessor(chunk_raw[feature_cols].copy(), preprocessor)

    # align columns in case missing indicators differ
    for col in transformed_columns:
        if col not in X_chunk.columns:
            X_chunk[col] = 0
    extra_cols = [c for c in X_chunk.columns if c not in transformed_columns]
    if extra_cols:
        X_chunk = X_chunk.drop(columns=extra_cols)
    X_chunk = X_chunk[transformed_columns]

    chunk_proba = model.predict_proba(X_chunk)[:, 1]
    chunk_pred = (chunk_proba >= FRAUD_THRESHOLD).astype(int)
    y_true = chunk_raw["isFraud"].astype(int).values

    # Chunk metrics
    chunk_pred_pos = int(chunk_pred.sum())
    chunk_true_pos = int(y_true.sum())

    try:
        chunk_auc = roc_auc_score(y_true, chunk_proba) if len(np.unique(y_true)) > 1 else np.nan
    except Exception:
        chunk_auc = np.nan

    try:
        chunk_ap = average_precision_score(y_true, chunk_proba) if len(np.unique(y_true)) > 1 else np.nan
    except Exception:
        chunk_ap = np.nan

    p, r, f1, _ = precision_recall_fscore_support(
        y_true, chunk_pred, average="binary", zero_division=0
    )

    # keep a few rows for the live display
    display_df = pd.DataFrame({
        "TransactionDT": chunk_raw["TransactionDT"].values,
        "fraud_probability": np.round(chunk_proba, 4),
        "predicted_class": chunk_pred,
        "actual_isFraud": y_true,
    }).head(10)
    latest_examples = display_df.copy()

    results.append({
        "chunk_idx": chunk_idx + 1,
        "rows_in_chunk": len(chunk_raw),
        "fraud_rate_chunk": float(np.mean(y_true)),
        "predicted_fraud_count": chunk_pred_pos,
        "actual_fraud_count": chunk_true_pos,
        "auc_chunk": chunk_auc,
        "ap_chunk": chunk_ap,
        "precision_chunk": p,
        "recall_chunk": r,
        "f1_chunk": f1,
        "training_rows_before_update": len(adaptive_train_df),
    })

    # reveal labels and update the available training set
    adaptive_train_df = pd.concat([adaptive_train_df, chunk_raw], axis=0, ignore_index=True)
    all_seen_stream_rows += len(chunk_raw)

    # optional rolling window to keep retraining fast
    if TRAINING_WINDOW is not None and len(adaptive_train_df) > TRAINING_WINDOW:
        adaptive_train_df = adaptive_train_df.iloc[-TRAINING_WINDOW:].copy()

    # retrain periodically
    if ((chunk_idx + 1) % RETRAIN_EVERY_CHUNKS == 0) or (chunk_idx == num_chunks_total - 1):
        model, preprocessor, transformed_columns = train_model(
            train_raw=adaptive_train_df,
            feature_cols=feature_cols,
            categorical_cols=categorical_cols,
            numerical_cols=numerical_cols,
        )
        retrain_count += 1

    # live-ish dashboard
    clear_output(wait=True)
    live_df = pd.DataFrame(results)
    print(f"Processed stream chunk {chunk_idx + 1}/{num_chunks_total}")
    print(f"Rows seen in live stream so far: {all_seen_stream_rows:,}")
    print(f"Model retrains completed: {retrain_count}")
    print("-" * 80)
    print("Latest chunk summary:")
    display(live_df.tail(5))
    print("-" * 80)
    print("Example scored transactions from latest chunk:")
    display(latest_examples)

    if SLEEP_SECONDS > 0:
        time.sleep(SLEEP_SECONDS)

results_df = pd.DataFrame(results)
print("Streaming simulation complete.")


## 7) Final summary metrics across the simulated stream

In [ ]:

results_df


In [ ]:

if not results_df.empty:
    weighted_fraud_rate = np.average(results_df["fraud_rate_chunk"], weights=results_df["rows_in_chunk"])
    weighted_precision = np.average(results_df["precision_chunk"], weights=results_df["rows_in_chunk"])
    weighted_recall = np.average(results_df["recall_chunk"], weights=results_df["rows_in_chunk"])
    weighted_f1 = np.average(results_df["f1_chunk"], weights=results_df["rows_in_chunk"])

    valid_auc = results_df["auc_chunk"].dropna()
    valid_ap = results_df["ap_chunk"].dropna()

    print("Weighted stream fraud rate:", round(weighted_fraud_rate, 4))
    print("Weighted stream precision:", round(weighted_precision, 4))
    print("Weighted stream recall:", round(weighted_recall, 4))
    print("Weighted stream F1:", round(weighted_f1, 4))
    print("Mean chunk AUC:", round(valid_auc.mean(), 4) if len(valid_auc) else "N/A (single-class chunks)")
    print("Mean chunk AP:", round(valid_ap.mean(), 4) if len(valid_ap) else "N/A (single-class chunks)")
else:
    print("No results were produced.")


## 8) Visualize how the model behaved over time

In [ ]:

if not results_df.empty:
    plt.figure(figsize=(10, 5))
    plt.plot(results_df["chunk_idx"], results_df["predicted_fraud_count"], marker="o")
    plt.xlabel("Chunk")
    plt.ylabel("Predicted fraud count")
    plt.title("Predicted Fraud Count by Incoming Chunk")
    plt.show()

    plt.figure(figsize=(10, 5))
    plt.plot(results_df["chunk_idx"], results_df["actual_fraud_count"], marker="o")
    plt.xlabel("Chunk")
    plt.ylabel("Actual fraud count")
    plt.title("Actual Fraud Count by Incoming Chunk")
    plt.show()

    if results_df["auc_chunk"].notna().sum() > 0:
        plt.figure(figsize=(10, 5))
        plt.plot(results_df["chunk_idx"], results_df["auc_chunk"], marker="o")
        plt.xlabel("Chunk")
        plt.ylabel("AUC")
        plt.title("Chunk-level AUC Over Time")
        plt.show()

    plt.figure(figsize=(10, 5))
    plt.plot(results_df["chunk_idx"], results_df["precision_chunk"], marker="o", label="Precision")
    plt.plot(results_df["chunk_idx"], results_df["recall_chunk"], marker="o", label="Recall")
    plt.plot(results_df["chunk_idx"], results_df["f1_chunk"], marker="o", label="F1")
    plt.xlabel("Chunk")
    plt.ylabel("Metric value")
    plt.title("Live Stream Classification Metrics Over Time")
    plt.legend()
    plt.show()


## 9) Optional: a single-transaction style demo view

If you want a more dramatic presentation, run this cell after the chunked simulation.  
It replays the first scored chunk row by row using the current model and prints what the model thinks each incoming transaction is.


In [ ]:

if len(stream_df) > 0:
    replay_n = min(25, len(stream_df))
    replay_df = stream_df.iloc[:replay_n].copy()

    X_replay = transform_with_preprocessor(replay_df[feature_cols].copy(), preprocessor)
    for col in transformed_columns:
        if col not in X_replay.columns:
            X_replay[col] = 0
    extra_cols = [c for c in X_replay.columns if c not in transformed_columns]
    if extra_cols:
        X_replay = X_replay.drop(columns=extra_cols)
    X_replay = X_replay[transformed_columns]

    replay_proba = model.predict_proba(X_replay)[:, 1]
    replay_pred = (replay_proba >= FRAUD_THRESHOLD).astype(int)

    replay_out = pd.DataFrame({
        "TransactionDT": replay_df["TransactionDT"].values,
        "fraud_probability": np.round(replay_proba, 4),
        "predicted_class": replay_pred,
        "actual_isFraud": replay_df["isFraud"].astype(int).values,
    })

    display(replay_out)
else:
    print("No stream rows available to replay.")


## 10) Talking points for the demo

You can say:

- “We first train on historical transactions only.”
- “The remaining transactions are treated as if they are arriving live.”
- “The model scores each new batch immediately.”
- “After labels become available, we fold those transactions into the training history.”
- “Then we retrain, so the model gradually adapts to the evolving stream.”

That is the clean supervised analogue of online learning for LightGBM.
